In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
df=pd.read_csv('dubai_properties.csv')
print(df.info())
print(df.describe())
df_filtered=df[['Rent', 'Beds', 'Baths', 'Type', 'Area_in_sqft', 'Furnishing', 'Location', 'City',]].copy()
df_filtered=df_filtered[df_filtered['Rent']>0]
df_filtered
lower_limit=df_filtered['Rent'].quantile(0.01)
upper_limit=df_filtered['Rent'].quantile(0.99)
df_filtered=df_filtered[(df_filtered['Rent']>=lower_limit) & (df_filtered['Rent']<=upper_limit)]
df_filtered.info()
lower_limit=df_filtered['Area_in_sqft'].quantile(0.01)
upper_limit=df_filtered['Area_in_sqft'].quantile(0.99)
df_filtered=df_filtered[(df_filtered['Area_in_sqft']>=lower_limit) & (df_filtered['Area_in_sqft']<=upper_limit)]
df_filtered.info()
df_filtered
y=np.log(df_filtered['Rent'])
x=df_filtered.drop(columns=['Rent'])
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape
preprocessor=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),['Beds','Baths','Area_in_sqft']),
        ('cat',OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Type','Furnishing','Location','City'])
    ]
)
model_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1))
])
model_pipeline.fit(x_train,y_train)
y_pred=np.exp(model_pipeline.predict(x_test))

y_test_actual=np.exp(y_test)

print(f"R2 SCORE: {r2_score(y_pred,y_test_actual)}")
print(f"MAE : {mean_absolute_error(y_pred,y_test_actual)}")
# Calculate the percentage error for every row
percentage_errors = np.abs((y_test_actual - y_pred) / y_test_actual) * 100

print(f"Median Error Percentage: {np.median(percentage_errors):.2f}%")
results_df = pd.DataFrame({'Actual': y_test_actual, 'Predicted': y_pred})
results_df
results_df['difference']=results_df['Predicted']-results_df['Actual']
results_df
# Let's inspect the exact row where the AI failed the hardest
print("--- The AI's Biggest Miss ---")
print(df_filtered.loc[31102])
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extract the actual model and the preprocessor from your pipeline
rf_model = model_pipeline.named_steps['model']
preprocessor = model_pipeline.named_steps['preprocessor']

# 2. Get the original numerical column names
num_cols = ['Beds', 'Baths', 'Area_in_sqft']

# 3. Extract the new categorical column names created by the OneHotEncoder
# We access the 'cat' transformer from the preprocessor step
cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(['Type', 'Furnishing', 'Location', 'City'])

# 4. Combine them into one master list of features
all_features = num_cols + list(cat_cols)

# 5. Get the importance scores from the Random Forest
importances = rf_model.feature_importances_

# 6. Create a DataFrame and sort it
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
})
importance_df = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

# 7. Plot the Top 15 most important features
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15), palette='viridis')
plt.title('Top 15 Drivers of Dubai Real Estate Prices (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# Print the top 5 to the console
print(importance_df.head(5))
import joblib
joblib.dump(model_pipeline,'price_prediction_model.pkl')
joblib.dump(list(x.columns),'model_columns.pkl')
x_train